https://www.geeksforgeeks.org/artificial-intelligence/agents-and-tools-in-langchain/


# 1. Calculator Tool

In [19]:
from langchain_core.tools import Tool

# 문자열로 된 수식을 계산하여 결과를 반환하는 함수
# expression: "2+2", "125 * 12" 같은 수학 수식 문자열
def calculator(expression: str) -> str:
    try:
        # eval()은 문자열을 파이썬 표현식으로 해석하고 실행
        # 예: eval("125 * 12") → 1500 (정수)
        # str()로 감싸서 반환 타입을 항상 문자열로 통일
        return str(eval(expression))
    except Exception as e:
        # 잘못된 수식(예: "abc + 1", "1 / 0") 입력 시 오류 메시지 반환
        # 예외를 던지지 않고 문자열로 반환해서 에이전트가 오류를 처리할 수 있게 함
        return f"Error: {e}"


# LangChain의 Tool 클래스로 함수를 에이전트가 사용할 수 있는 도구로 래핑
# 에이전트는 name과 description을 보고 어떤 상황에 이 도구를 쓸지 판단함
calc_tool = Tool(
    name="Calculator",      # 도구의 고유 이름 - 에이전트가 도구를 호출할 때 이 이름으로 참조
    func=calculator,        # 도구가 실행할 실제 함수 - Tool.run() 호출 시 이 함수가 실행됨
    description="Evaluate a math expression like 2+2"
    # description은 LLM이 읽는 텍스트 - 언제 이 도구를 써야 하는지 명확히 설명할수록 좋음
    # 예: "숫자 계산이 필요할 때 사용. 사칙연산, 괄호 표현식 지원" 처럼 구체적으로 작성 권장
)

# calc_tool.run()으로 도구 직접 실행 (에이전트 없이 단독 테스트)
# 내부적으로 func=calculator 에 "125 * 12" 를 전달 → calculator("125 * 12") 호출
# 결과: "1500" (문자열)
result = calc_tool.run("125 * 12")

# 2. Web Search Tool

## 무료 Tool

In [5]:
! uv pip install -U ddgs

Using Python 3.12.12 environment at: C:\Users\jino7\Desktop\llm_engineering-main\.venv
Resolved 18 packages in 161ms
 Downloaded primp
 Downloaded lxml
Prepared 9 packages in 187ms
Uninstalled 4 packages in 31ms
Installed 9 packages in 161ms
 - anyio==4.11.0
 + anyio==4.13.0
 - certifi==2025.10.5
 + certifi==2026.5.20
 - click==8.3.0
 + click==8.4.1
 + ddgs==9.14.4
 + fake-useragent==2.2.0
 - idna==3.11
 + idna==3.18
 + lxml==6.1.1
 + primp==1.3.1
 + socksio==1.0.0


In [20]:
from langchain_community.tools import DuckDuckGoSearchRun  # DuckDuckGo 웹 검색 도구 (API 키 불필요)
from langchain_core.tools import Tool                       # 함수를 에이전트 도구로 래핑하는 클래스

# DuckDuckGoSearchRun 인스턴스 생성
# - API 키 없이 무료로 웹 검색 가능
# - 내부적으로 DuckDuckGo 검색 엔진에 HTTP 요청을 보내 결과를 텍스트로 반환
# - 단, 요청이 잦으면 일시적으로 차단될 수 있음 (rate limit 존재)
search = DuckDuckGoSearchRun()

# DuckDuckGo 검색 기능을 LangChain Tool로 래핑
# 에이전트가 "웹 검색이 필요한 상황"임을 판단하면 이 도구를 자동으로 선택
search_tool = Tool(
    name="Web Search",          # 에이전트가 도구를 식별하는 이름
    func=search.run,            # DuckDuckGoSearchRun의 run 메서드를 직접 연결
                                # search.run("쿼리") → 검색 결과 문자열 반환
    description="Search the web and return relevant results"
                                # LLM이 이 텍스트를 읽고 도구 사용 여부를 판단
                                # 더 구체적으로 작성할수록 에이전트의 도구 선택 정확도가 높아짐
                                # 예: "최신 뉴스, 날씨, 실시간 정보 검색이 필요할 때 사용"
)

# 도구 직접 실행 (에이전트 없이 단독 테스트)
# search_tool.run() → 내부적으로 search.run() → DuckDuckGo 검색 실행
# 반환값: 검색 결과를 하나의 긴 문자열로 합친 텍스트
result = search_tool.run("today is weather in korea?")

# 검색 결과 전체는 매우 길기 때문에 앞 200자만 출력해서 확인
# result[:200] → 문자열 슬라이싱으로 처음 200자만 추출
print("Web Search Tool Output:", result[:200], "...")

Web Search Tool Output: Weather. You can register the area of interest after searching the area. Eup/Myeon/Dong search.Current Weather in Korea. Short-Range Forecasts. Current local time in South Korea – Seoul. Get Seoul's w ...


## [TavilySearch]

In [17]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [21]:
from langchain_community.tools.tavily_search import TavilySearchResults
# Tavily: AI 특화 검색 엔진으로, DuckDuckGo보다 구조화된 결과를 반환
# 환경변수 TAVILY_API_KEY 설정 필요 (https://tavily.com 에서 발급)

# 고객지원 질문 검색용 Tavily 인스턴스 생성
support_search = TavilySearchResults(
    max_results=3,             # 검색 결과를 최대 3개 문서로 제한
                               # 너무 많으면 LLM 컨텍스트 초과 위험 → 3~5개 권장
    include_answer=True,       # Tavily가 자체적으로 생성한 요약 답변 포함
                               # True 시 result에 'answer' 키가 추가됨
    include_raw_content=True,  # 각 검색 결과의 원문 전체를 포함
                               # False면 스니펫(일부 발췌)만 반환, True면 전체 본문 포함
                               # RAG 파이프라인에서 청킹/임베딩 용도로 활용할 때 유용
)

support_search  # 객체 구조 확인 (Jupyter 환경에서 속성값 출력)

# Tavily 검색 실행
# invoke()는 LangChain의 표준 실행 메서드 (run() 대신 LCEL 호환 방식으로 권장됨)
# 반환값: List[Dict] 형태
# [
#   {
#     "url": "https://...",       # 출처 URL
#     "content": "...",           # 본문 내용 (include_raw_content=True면 전체 원문)
#     "score": 0.98,              # Tavily의 관련도 점수 (1에 가까울수록 관련성 높음)
#     "answer": "..."             # 요약 답변 (include_answer=True일 때만 존재)
#   },
#   ...
# ]
cs_search_docs = support_search.invoke("교환과 환불 정책의 차이는?")

cs_search_docs  # 검색 결과 확인 (Jupyter 환경에서 리스트 구조 출력)
# 👉 검색 실행 (AI가 참고할 수 있는 자료 수집)

[{'title': '17. 교환/반품/환불 정책 정의',
  'url': 'https://hyeyun133.tistory.com/55',
  'content': "랑콤 교환정책\n\n아래 육육걸즈의 경우도 교환이 아닌 재주문을 해야 하는데 이 때 결제 금액 자체도 환불처리하지 않고 예치금으로 적립시켜 버린다. 아무래도 결제수단에 따라 환불이 완료되기까지 1~3일 정도의 시간이 소요되므로 교환과 환불 사이의 타협점을 찾은 것으로 보인다. 어쨌든 이 방법도 결국 재주문을 넣는 것이기 때문에 정확한 의미의 상품 '교환'은 아니다.\n\n육육걸즈 교환정책\n\n그 외 보통 쇼핑몰의 교환절차는 게시글에 직접 상품명과 옵션을 기재하여 교환신청을 넣는다. 이 때 상품을 반송하고(2,500원) 다른 상품으로 교환(2,500원)하므로 교환 시 배송비는 5,000원이 부담된다. 배송비를 지불하는 방법에는 반품하는 상품에 현금을 동봉하거나, 선입금하고 교환신청 시 입금자명을 기재하거나, 환불 시 차감하는 방법이 있다.\n\n이렇게 교환 상품을 기재하는 경우는 다른 상품, 다른 옵션으로도 교환 가능하므로 소비자 입장에서 가장 교환에 제한이 없지만 CS에서 교환 건마다 직접 교환 상품을 지정해주어야 한다는 단점이 있다.\n\n플라이모델 교환정책\n\n관리자가 '교환 후 상품'을 직접 지정해주어야 한다\n\n반면 소비자의 자유도를 낮추고 CS CAPA에 따라 같은 상품의 다른 옵션으로만 교환하도록 제한한다면 무신사의 방법이 가장 절충안이라고 볼 수 있다. 무신사의 경우는 교환 요청 시 상품 옵션을 변경할 수 있고 만약 출고 이후라면 배송비도 바로 결제할 수 있다.\n\n무신사 교환정책\n\n#### '서비스 기획 > 바닥부터 이커머스 시스템 기획' 카테고리의 다른 글 [...] # 중급닌자 연습생의 삽질하는 공간\n\n블로그 이미지\n\n# 17. 교환/반품/환불 정책 정의\n\n1단계: 처리상태 정의하기\n\n- 주문취소 상태에 대해 정의했다. (입금전/결제완료/상품준비중(배송전)

# 3. PDF Reader Tool

In [22]:
from langchain_community.document_loaders import PyPDFLoader  # PDF 파일을 페이지 단위로 읽는 로더
from langchain_core.tools import Tool                          # 함수를 에이전트 도구로 래핑하는 클래스

# PDF 파일 경로를 입력받아 전체 텍스트를 하나의 문자열로 반환하는 함수
def read_pdf(file_path: str) -> str:
    # PyPDFLoader: pypdf 라이브러리 기반으로 PDF를 페이지별 Document 객체로 파싱
    # 설치 필요: pip install pypdf
    loader = PyPDFLoader(file_path)

    # loader.load() → List[Document] 반환
    # 각 Document는 하나의 페이지에 해당하며 아래 속성을 가짐
    # - page_content: 해당 페이지의 텍스트 내용 (str)
    # - metadata: {"source": "파일경로", "page": 페이지번호} (dict)
    docs = loader.load()

    # 모든 페이지의 텍스트를 줄바꿈으로 연결해 하나의 문자열로 합침
    # 예: [페이지1 텍스트, 페이지2 텍스트, ...] → "페이지1\n페이지2\n..."
    # ⚠️ 페이지 구분 없이 합치므로 페이지 번호가 필요하면 metadata["page"] 활용 권장
    return "\n".join([page.page_content for page in docs])


# PDF 읽기 함수를 LangChain Tool로 래핑
# 에이전트가 "PDF 파일 내용이 필요한 상황"이라 판단하면 이 도구를 선택
pdf_tool = Tool(
    name="PDF Reader",      # 에이전트가 도구를 식별하는 이름
    func=read_pdf,          # Tool.run() 호출 시 실행될 함수
                            # read_pdf(file_path) 형태로 호출됨
    description="Read a PDF file and return its text content"
                            # LLM이 읽는 도구 설명 - 언제 이 도구를 써야 할지 판단 근거
                            # 예: "로컬 PDF 파일의 텍스트를 추출할 때 사용. 입력값: 파일 경로(str)"
                            #      처럼 입력 형식을 명시하면 에이전트 정확도가 높아짐
)

# 도구 직접 실행 - "test.pdf" 파일을 읽어 전체 텍스트 반환
# ⚠️ 파일이 존재하지 않으면 FileNotFoundError 발생
# ⚠️ 스캔된 PDF(이미지 기반)는 텍스트 추출 불가 → OCR 도구(예: pytesseract) 별도 필요
result = pdf_tool.run("test.pdf")

# 전체 텍스트는 매우 길 수 있으므로 앞 100자만 잘라서 출력
# result[:100] → 문자열 슬라이싱으로 처음 100자만 추출
print("PDF Reader Tool Output:", result[:100], "...")

PDF Reader Tool Output: 2022 
경기 규칙

LGFA PLAYING Rules 2022 1 
  
 
 
 
 
 
 
 
소개 
본 소책자는 여자 게일식 축구의 경기 규칙을 간략하게 
설명하고 있습니 ...


# 4. Python REPL Tool:

In [23]:
from langchain_core.tools import Tool  # 함수를 에이전트 도구로 래핑하는 클래스

# 파이썬 코드 문자열을 입력받아 실행 결과를 반환하는 함수
# eval()을 사용하므로 단일 표현식(expression)만 실행 가능
# 예: "10 + 25" ✅  /  "x = 10\nprint(x)" ❌ (여러 줄 문은 exec() 필요)
def python_eval(code: str) -> str:
    try:
        # eval(): 문자열을 파이썬 표현식으로 해석하고 즉시 실행
        # 반환값이 정수, 리스트 등 다양한 타입일 수 있으므로 str()로 통일
        return str(eval(code))
    except Exception as e:
        # 잘못된 코드 입력 시 예외를 던지지 않고 오류 메시지를 문자열로 반환
        # 에이전트가 오류 내용을 읽고 재시도하거나 다른 도구를 선택할 수 있게 함
        return f"Error: {e}"

# ⚠️ 보안 경고: eval()은 입력된 코드를 제한 없이 실행하므로 매우 위험
# 악의적인 입력 예: eval("__import__('os').system('rm -rf /')")
# 실제 서비스에서는 아래 대안 사용 권장:
# 1. langchain_experimental의 PythonREPLTool (샌드박스 환경 제공)
# 2. RestrictedPython 라이브러리로 실행 가능한 코드 범위 제한
# 3. Docker 컨테이너 등 격리된 환경에서 실행


# 파이썬 코드 실행 기능을 LangChain Tool로 래핑
python_tool = Tool(
    name="Python REPL",     # 에이전트가 도구를 식별하는 이름
                            # REPL: Read-Eval-Print Loop (대화형 실행 환경)
    func=python_eval,       # Tool.run() 호출 시 실행될 함수
    description="Run Python code and return the result"
                            # LLM이 읽는 도구 설명 - 수식 계산, 데이터 처리 등
                            # 수치 계산이 필요할 때 Calculator 대신 이 도구를 선택할 수 있음
                            # 예: "수식 계산, 리스트 조작, 문자열 처리 등 파이썬 표현식 실행 시 사용"
                            #      처럼 구체적으로 작성하면 에이전트 도구 선택 정확도 향상
)

# 도구 직접 실행 테스트
# python_tool.run("10 + 25") → python_eval("10 + 25") → eval("10 + 25") → 35 → "35"
result = python_tool.run("10 + 25")

# 결과 출력: "Python REPL Tool Output: 35"
print("Python REPL Tool Output:", result)

Python REPL Tool Output: 35


# 5. Custom Tool 만들기

In [24]:
# Custom Tool 만들기
from langchain_core.tools import tool  # 👉 @tool 데코레이터 불러오기

@tool
def sum_counts(open_tickets: int, solved_today: int) -> int:
    "접수 티켓 수와 당일 해결 티켓 수를 합산해 총 건수를 반환합니다."
    return open_tickets + solved_today
# 👉 간단한 합산 도구 (예: 티켓 현황 계산기)

@tool
def reverse_sentence(passage: str) -> str:
    "입력된 문장을 뒤집어 반환합니다."
    return passage[::-1]
# 👉 문자열 뒤집기 도구 (예: 텍스트 가공)

print(sum_counts.invoke({'open_tickets': 12, 'solved_today': 5}))
print(reverse_sentence.invoke({'passage': '고객 응대는 친절이 핵심'}))
# 👉 사용자 정의 툴 직접 실행해보기

17
심핵 이절친 는대응 객고
